# Détection de Textes Générés par l'IA
**Barbara KENGNE — Licence 3 Informatique, Aix-Marseille Université**

Avec la multiplication des LLMs (GPT-4, Claude, Gemini...), distinguer un texte écrit par un humain d'un texte généré par une IA devient un enjeu majeur.  
Ce projet explore les caractéristiques stylistiques qui différencient ces textes, et construit un classifieur capable de les détecter.

---

## 1. Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')
print('✅ Bibliothèques importées')

## 2. Chargement et exploration des données

In [ ]:
df = pd.read_csv('AuthentiText_X_2026_AI_vs_Human_Detection_1K.csv')
print(f'Dataset : {df.shape[0]} textes, {df.shape[1]} colonnes')
print(f'\nColonnes : {df.columns.tolist()}')
df.head(3)

In [ ]:
print('Distribution AI vs Human :')
print(df['author_type'].value_counts())
print('\nDistribution par modèle source :')
print(df['model_source'].value_counts())

In [ ]:
# Visualisation de la distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# AI vs Human
counts = df['author_type'].value_counts()
colors = ['#e74c3c', '#2ecc71']
axes[0].pie(counts, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Répartition AI vs Human', fontsize=13, fontweight='bold')

# Par modèle source
source_counts = df['model_source'].value_counts()
palette = ['#3498db', '#9b59b6', '#e67e22', '#2ecc71']
bars = axes[1].bar(source_counts.index, source_counts.values, color=palette, edgecolor='white')
axes[1].set_title('Répartition par modèle source', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Nombre de textes')
for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., h + 3, str(int(h)), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Analyse exploratoire — Qu'est-ce qui différencie les textes IA des textes humains ?

In [ ]:
# Comparaison des features numériques entre AI et Human
features_num = [
    'perplexity_score', 'burstiness_index', 'syntactic_variability',
    'semantic_coherence_score', 'lexical_diversity_ratio', 'readability_grade_level'
]

print('Moyennes par catégorie (AI vs Human) :\n')
print(df.groupby('author_type')[features_num].mean().round(3).T.to_string())

In [ ]:
# Visualisation des distributions pour chaque feature
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

colors_map = {'AI': '#e74c3c', 'Human': '#2ecc71'}

for i, feature in enumerate(features_num):
    for label, color in colors_map.items():
        subset = df[df['author_type'] == label][feature]
        axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='white')
    axes[i].set_title(feature.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Valeur')
    axes[i].set_ylabel('Fréquence')
    axes[i].legend()

plt.suptitle('Distribution des features stylistiques : AI vs Human', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('features_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap de corrélation
fig, ax = plt.subplots(figsize=(9, 6))
corr = df[features_num].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Corrélation entre les features stylistiques', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analyse par modèle : chaque IA a-t-elle son propre style ?
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

palette_models = {'Gemini': '#3498db', 'Claude': '#9b59b6', 'GPT-4': '#e67e22', 'Human': '#2ecc71'}

for ax, feat in zip(axes, ['burstiness_index', 'lexical_diversity_ratio']):
    for model, color in palette_models.items():
        subset = df[df['model_source'] == model][feat]
        ax.hist(subset, bins=25, alpha=0.55, color=color, label=model, edgecolor='white')
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Valeur')
    ax.legend()

plt.suptitle('Comparaison par modèle source', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparaison_modeles_source.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Approche 1 — Classification sur les features numériques

In [ ]:
# Encodage de la cible
df['label'] = (df['author_type'] == 'AI').astype(int)  # AI=1, Human=0

X_num = df[features_num].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_num, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train : {len(X_train)} | Test : {len(X_test)}')

In [ ]:
modeles_num = {
    'Régression Logistique': LogisticRegression(max_iter=1000),
    'Random Forest':         RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM Linéaire':          LinearSVC(max_iter=2000)
}

resultats_num = []
for nom, modele in modeles_num.items():
    modele.fit(X_train_sc, y_train)
    y_pred = modele.predict(X_test_sc)
    resultats_num.append({
        'Modèle':    nom,
        'Accuracy':  round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test, y_pred) * 100, 2),
        'Recall':    round(recall_score(y_test, y_pred) * 100, 2),
        'F1-Score':  round(f1_score(y_test, y_pred) * 100, 2)
    })

df_num = pd.DataFrame(resultats_num)
print('📊 Résultats avec features numériques :\n')
print(df_num.to_string(index=False))

## 5. Approche 2 — Classification sur le texte brut (TF-IDF)

In [ ]:
X_text = df['content_text']

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train_t)
X_test_tfidf  = tfidf.transform(X_test_t)

print(f'Matrice TF-IDF : {X_train_tfidf.shape}')

In [ ]:
modeles_text = {
    'Naive Bayes':           MultinomialNB(),
    'Régression Logistique': LogisticRegression(max_iter=1000),
    'SVM Linéaire':          LinearSVC(max_iter=2000)
}

resultats_text = []
for nom, modele in modeles_text.items():
    modele.fit(X_train_tfidf, y_train_t)
    y_pred = modele.predict(X_test_tfidf)
    resultats_text.append({
        'Modèle':    nom,
        'Accuracy':  round(accuracy_score(y_test_t, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test_t, y_pred) * 100, 2),
        'Recall':    round(recall_score(y_test_t, y_pred) * 100, 2),
        'F1-Score':  round(f1_score(y_test_t, y_pred) * 100, 2)
    })

df_text = pd.DataFrame(resultats_text)
print('📊 Résultats avec TF-IDF (texte brut) :\n')
print(df_text.to_string(index=False))

## 6. Comparaison des deux approches

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df_res, titre) in zip(axes, [
    (df_num,  'Features numériques'),
    (df_text, 'TF-IDF (texte brut)')
]):
    x = np.arange(len(df_res))
    width = 0.2
    metriques = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    colors_m = ['#3498db', '#9b59b6', '#e67e22', '#1abc9c']
    for i, (m, c) in enumerate(zip(metriques, colors_m)):
        bars = ax.bar(x + i * width, df_res[m], width, label=m, color=c, alpha=0.85)
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., h + 0.3,
                    f'{h:.0f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(df_res['Modèle'], fontsize=9)
    ax.set_ylim(40, 108)
    ax.set_ylabel('Score (%)')
    ax.set_title(f'Approche : {titre}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Comparaison des deux approches de classification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparaison_approches.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Analyse : quels mots trahissent un texte IA ?

In [ ]:
# Mots les plus caractéristiques des textes IA vs humains
# via les coefficients de la Régression Logistique
lr_model = modeles_text['Régression Logistique']
feature_names = tfidf.get_feature_names_out()
coefs = lr_model.coef_[0]

top_n = 15
top_ai    = np.argsort(coefs)[-top_n:][::-1]
top_human = np.argsort(coefs)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mots IA
axes[0].barh(range(top_n), coefs[top_ai][::-1], color='#e74c3c', alpha=0.8)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(feature_names[top_ai][::-1], fontsize=10)
axes[0].set_title('Mots les plus associés aux textes IA 🤖', fontweight='bold')
axes[0].set_xlabel('Coefficient')

# Mots humains
axes[1].barh(range(top_n), np.abs(coefs[top_human])[::-1], color='#2ecc71', alpha=0.8)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(feature_names[top_human][::-1], fontsize=10)
axes[1].set_title('Mots les plus associés aux textes Humains ✍️', fontweight='bold')
axes[1].set_xlabel('|Coefficient|')

plt.suptitle('Quels mots trahissent un texte IA ?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mots_trahisseurs.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Matrice de confusion du meilleur modèle

In [ ]:
# Meilleur modèle selon F1-Score
tous_resultats = pd.concat([
    df_num.assign(Approche='Features numériques'),
    df_text.assign(Approche='TF-IDF')
])
meilleur = tous_resultats.loc[tous_resultats['F1-Score'].idxmax()]
print(f'Meilleur modèle : {meilleur["Modèle"]} ({meilleur["Approche"]})')
print(f'F1-Score : {meilleur["F1-Score"]}%')

# Récupération du modèle
if meilleur['Approche'] == 'TF-IDF':
    best_model = modeles_text[meilleur['Modèle']]
    y_pred_best = best_model.predict(X_test_tfidf)
    y_test_best = y_test_t
else:
    best_model = modeles_num[meilleur['Modèle']]
    y_pred_best = best_model.predict(X_test_sc)
    y_test_best = y_test

cm = confusion_matrix(y_test_best, y_pred_best)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=['Human', 'AI'],
            yticklabels=['Human', 'AI'],
            linewidths=0.5, ax=ax)
ax.set_xlabel('Prédiction', fontsize=12)
ax.set_ylabel('Réalité', fontsize=12)
ax.set_title(f'Matrice de confusion\n{meilleur["Modèle"]} ({meilleur["Approche"]})', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 Rapport complet :')
print(classification_report(y_test_best, y_pred_best, target_names=['Human', 'AI']))

## 9. Tester sur un texte personnalisé

In [ ]:
def predire_origine(texte):
    """Prédit si un texte a été écrit par une IA ou un humain."""
    vecteur = tfidf.transform([texte])
    pred = modeles_text['SVM Linéaire'].predict(vecteur)[0]
    label = '🤖 Texte IA' if pred == 1 else '✍️ Texte Humain'
    print(f'Texte : "{texte[:80]}..."')
    print(f'Résultat : {label}\n')

# Tests
predire_origine("In conclusion, it is important to note that the aforementioned factors contribute significantly to the overall outcome of the situation.")
predire_origine("honestly idk man, i just felt like it made more sense to do it that way lol")
predire_origine("Furthermore, leveraging synergistic approaches can optimize the efficiency of the implemented solutions.")

## 10. Conclusion

Ce projet explore la détection de textes générés par IA à travers deux approches complémentaires :

**Approche 1 — Features stylistiques numériques** (perplexité, burstiness, diversité lexicale...)
- Avantage : interprétable, rapide
- Limite : dépend de la qualité des features extraites

**Approche 2 — TF-IDF sur le texte brut**
- Avantage : capture les patterns lexicaux réels
- L'analyse des coefficients révèle les mots "trahisseurs" des textes IA

**Observation intéressante :** les textes de Gemini, Claude et GPT-4 présentent des profils stylistiques légèrement différents, ce qui suggère qu'une classification multi-classe par modèle source serait envisageable.

**Pistes d'amélioration :**
- Utiliser des embeddings (BERT, RoBERTa) pour une représentation sémantique plus riche
- Tester une classification multi-classe (Human / GPT-4 / Claude / Gemini)
- Collecter des données plus longues pour améliorer la robustesse